In [501]:
pip install pandas scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [502]:
import pandas as pd 

df1 = pd.read_csv("uci", sep="\t", header=None)
df1.columns = ["label", "message"]

print(df1.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [503]:
df2 = pd.read_csv("syn.csv")
df2.columns = ["label", "message"]

print(df2.head())

  label                                            message
0  spam  Electricity bill pending of Rs 5000. Pay withi...
1  spam     KYC incomplete for HDFC. Submit documents now.
2   ham                                    Happy Birthday!
3  spam  Job offer confirmed. Pay registration fee Rs 5...
4  spam         PAN not linked with Aadhaar. Update today.


In [504]:
df= pd.concat([df1, df2])
print(df)

      label                                            message
0       ham  Go until jurong point, crazy.. Available only ...
1       ham                      Ok lar... Joking wif u oni...
2      spam  Free entry in 2 a wkly comp to win FA Cup fina...
3       ham  U dun say so early hor... U c already then say...
4       ham  Nah I don't think he goes to usf, he lives aro...
...     ...                                                ...
11995  spam  KYC incomplete for HDFC. Submit documents imme...
11996  spam  Electricity bill pending of Rs 2500000. Pay wi...
11997  spam  KYC incomplete for Bank of Baroda. Submit docu...
11998  spam  RBI alert: Suspicious activity detected. Verif...
11999   ham                   I will reach home in 20 minutes.

[17572 rows x 2 columns]


In [505]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_message'] = df['message'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abhia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [506]:
df

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though
...,...,...,...
11995,spam,KYC incomplete for HDFC. Submit documents imme...,kyc incomplete hdfc submit documents immediately
11996,spam,Electricity bill pending of Rs 2500000. Pay wi...,electricity bill pending rs 2500000 pay within...
11997,spam,KYC incomplete for Bank of Baroda. Submit docu...,kyc incomplete bank baroda submit documents ur...
11998,spam,RBI alert: Suspicious activity detected. Verif...,rbi alert suspicious activity detected verify ...


In [507]:
df = df.dropna()
df = df.reset_index(drop=True)

In [508]:
df

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though
...,...,...,...
17567,spam,KYC incomplete for HDFC. Submit documents imme...,kyc incomplete hdfc submit documents immediately
17568,spam,Electricity bill pending of Rs 2500000. Pay wi...,electricity bill pending rs 2500000 pay within...
17569,spam,KYC incomplete for Bank of Baroda. Submit docu...,kyc incomplete bank baroda submit documents ur...
17570,spam,RBI alert: Suspicious activity detected. Verif...,rbi alert suspicious activity detected verify ...


In [509]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# vectorizer = TfidfVectorizer(
#     max_features=800,     
#     ngram_range=(1,2)     
# )

# X = vectorizer.fit_transform(df['clean_message'])

# print(X.shape)   

In [510]:
# vectorizer.get_feature_names_out()

In [511]:
# from sklearn.preprocessing import LabelEncoder

# le = LabelEncoder()
# y = le.fit_transform(df['label'])  # spam=1, ham=0

In [512]:
# from sklearn.preprocessing import MinMaxScaler

# scaler = MinMaxScaler()
# X_norm = scaler.fit_transform(X.toarray())

In [513]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

In [514]:
from sklearn.model_selection import train_test_split

X_text = df['clean_message']
y = df['label']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

In [515]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=1200,
    ngram_range=(1,3),
    stop_words="english",
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

In [516]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train.toarray())
X_test = scaler.transform(X_test.toarray())

In [517]:
from sklearn.neural_network import MLPClassifier

ann_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),   # 2 hidden layers
    activation='relu',
    solver='adam',
    max_iter=20,
    random_state=42
)

ann_model.fit(X_train, y_train)

c:\Users\abhia\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


,hidden_layer_sizes,"(128, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,20
,shuffle,True
,random_state,42


In [518]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred =ann_model.predict_proba(X_test)[:,1]

y_pred = (y_pred > 0.4).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9890453834115805
Precision: 0.9905030800821355
Recall: 0.9897409592203129
F1-score: 0.9901218729955099

Detailed Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3130
           1       0.99      0.99      0.99      3899

    accuracy                           0.99      7029
   macro avg       0.99      0.99      0.99      7029
weighted avg       0.99      0.99      0.99      7029



In [519]:
total_params = 0

for coef, intercept in zip(ann_model.coefs_, ann_model.intercepts_):
    total_params += coef.size      # weights
    total_params += intercept.size # bias

print("Total ANN Parameters:", total_params)

Total ANN Parameters: 162049


## SnnTorch

In [520]:
pip install torch snntorch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [521]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert train/test data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [522]:
def latency_encoding(x, T=10):
    """
    x: (batch, features)
    returns: (T, batch, features)
    """
    batch_size, num_features = x.shape
    
    spike_times = (T * (1 - x)).long()   # compute spike time
    
    spikes = torch.zeros(T, batch_size, num_features)
    
    for t in range(T):
        spikes[t] = (spike_times == t).float()
        
    return spikes

In [523]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

beta = 0.9
spike_grad = surrogate.fast_sigmoid()

class SNNModel(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)

        self.fc2 = nn.Linear(hidden_size, output_size)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk2_rec = []

        num_steps = x.size(0)

        for step in range(num_steps):

            cur1 = self.fc1(x[step])
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk2_rec.append(spk2)

        return torch.stack(spk2_rec)

In [524]:
input_size = X_norm.shape[1]

print("Input features:", input_size)

Input features: 800


In [525]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = 1200

model = SNNModel(
    input_size=input_size,
    hidden_size=256,
    output_size=2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [526]:
num_epochs = 25
T = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)
        
        spikes = latency_encoding(data, T).to(device)
        
        optimizer.zero_grad()
        
        outputs = model(spikes)
        
        # sum spikes over time
        outputs = outputs.sum(dim=0)
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.4381
Epoch 2, Loss: 0.0493
Epoch 3, Loss: 0.0326
Epoch 4, Loss: 0.0280
Epoch 5, Loss: 0.0235
Epoch 6, Loss: 0.0219
Epoch 7, Loss: 0.0222
Epoch 8, Loss: 0.0195
Epoch 9, Loss: 0.0187
Epoch 10, Loss: 0.0164
Epoch 11, Loss: 0.0148
Epoch 12, Loss: 0.0172
Epoch 13, Loss: 0.0171
Epoch 14, Loss: 0.0143
Epoch 15, Loss: 0.0127
Epoch 16, Loss: 0.0122
Epoch 17, Loss: 0.0108
Epoch 18, Loss: 0.0117
Epoch 19, Loss: 0.0116
Epoch 20, Loss: 0.0101
Epoch 21, Loss: 0.0118
Epoch 22, Loss: 0.0101
Epoch 23, Loss: 0.0110
Epoch 24, Loss: 0.0094
Epoch 25, Loss: 0.0108


In [527]:
from sklearn.metrics import classification_report

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)
        targets = targets.to(device)
        
        spikes = latency_encoding(data, T).to(device)
        outputs = model(spikes)
        outputs = outputs.sum(dim=0)
        
        preds = torch.argmax(outputs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(classification_report(all_targets, all_preds))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3130
           1       1.00      0.99      0.99      3899

    accuracy                           0.99      7029
   macro avg       0.99      0.99      0.99      7029
weighted avg       0.99      0.99      0.99      7029



In [528]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for data, targets in test_loader:
        data = data.to(device)
        targets = targets.to(device)

        spikes = latency_encoding(data, T).to(device)
        outputs = model(spikes)
        outputs = outputs.sum(dim=0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# Convert to numpy
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Metrics
print("Accuracy:", accuracy_score(all_targets, all_preds))
print("Precision:", precision_score(all_targets, all_preds))
print("Recall:", recall_score(all_targets, all_preds))
print("F1-score:", f1_score(all_targets, all_preds))

print("\nDetailed Report:\n")
print(classification_report(all_targets, all_preds))

Accuracy: 0.9893299189073836
Precision: 0.9953367875647668
Recall: 0.9853808668889459
F1-score: 0.9903338059028225

Detailed Report:

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3130
           1       1.00      0.99      0.99      3899

    accuracy                           0.99      7029
   macro avg       0.99      0.99      0.99      7029
weighted avg       0.99      0.99      0.99      7029



In [529]:
print("----- MODEL COMPARISON -----")
print("ANN F1  : 0.9899717150938545 ")
print("SNN F1  :", f1_score(all_targets, all_preds))

----- MODEL COMPARISON -----
ANN F1  : 0.9899717150938545 
SNN F1  : 0.9903338059028225


In [530]:
# Evaluate on training set
model.eval()
train_preds = []
train_targets = []

with torch.no_grad():
    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)
        
        spikes = latency_encoding(data, T).to(device)
        outputs = model(spikes)
        outputs = outputs.sum(dim=0)
        
        preds = torch.argmax(outputs, dim=1)
        
        train_preds.extend(preds.cpu().numpy())
        train_targets.extend(targets.cpu().numpy())

print("Train F1:", f1_score(train_targets, train_preds))

Train F1: 0.9977724468814256


In [531]:
total_params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", total_params)

Total Parameters: 307970


In [532]:
torch.save(model.state_dict(), "snn_spam_model.pth")

In [533]:
import joblib

joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']